RAG con índice documental y un SLM
==================================

En el ejemplo anterior vimos el corazón de RAG: embeddings, similitud y contexto. Sin embargo, utilizamos documentos de texto sencillos y generamos los embeddings de forma manual almacenandonos en una lista en memoria. Tal técnica no puede escalar a grandes repositorios de documentos como los que podemos encontrar en una organización.

En este ejemplo, vamos a dar un paso más hacia una arquitectura de trabajo más parecida a la que encontraríamos en una organización: documentos con metadatos, chunking con solapamiento, un índice vectorial y un modelo generativo que responde usando los fragmentos recuperados.

## Preparación del ambiente

Instalemos las librerías necesarias desde el archivo de requerimientos asociado. En Google Colab puede ser necesario reiniciar la sesión después de la instalación si el entorno lo solicita.

In [ ]:
!wget -q https://raw.githubusercontent.com/santiagxf/M72109/master/docs/document-understanding/rag-index-slm.txt
%pip install -r rag-index-slm.txt --quiet

## Documentos de ejemplo

Para mantener el ejemplo controlado vamos a simular una pequeña base documental interna. Note que cada documento tiene texto, pero también metadatos: fuente, área y fecha. En una arquitectura real estos metadatos son muy importantes para filtrar resultados, auditar respuestas y aplicar controles de acceso.

In [ ]:
documentos_empresa = [
    {
        "fuente": "politica_datos_ia.md",
        "area": "gobierno de datos",
        "fecha": "2026-01-15",
        "texto": """
        La política de datos para proyectos de inteligencia artificial establece que todo conjunto de datos con información sensible debe pasar por clasificación, minimización y revisión de privacidad antes de ser usado. Los equipos deben conservar evidencia del consentimiento o de la base legal aplicable. Cuando se usen datos personales, se recomienda anonimización o seudonimización antes de crear embeddings. Los accesos al índice deben auditarse y limitarse según el rol del usuario.
        """,
    },
    {
        "fuente": "manual_soporte_clientes.md",
        "area": "operaciones",
        "fecha": "2026-02-03",
        "texto": """
        El equipo de soporte debe responder incidentes críticos en menos de cuatro horas hábiles. Si el incidente afecta a más de cincuenta clientes, se debe escalar al responsable de continuidad operativa. Las respuestas al cliente deben incluir número de ticket, estado actual y próximo paso. No se deben compartir detalles internos de infraestructura en comunicaciones externas.
        """,
    },
    {
        "fuente": "guia_rag_productivo.md",
        "area": "arquitectura",
        "fecha": "2026-03-10",
        "texto": """
        Una solución RAG productiva debe separar la ingesta documental de la consulta en línea. La ingesta extrae texto, limpia contenido, divide en chunks, calcula embeddings y publica los resultados en un índice vectorial. La consulta valida permisos, recupera fragmentos relevantes, opcionalmente aplica reranking, construye el prompt y registra trazas para evaluación. Se recomienda medir calidad de retrieval, fidelidad de la respuesta y tasa de respuestas sin evidencia suficiente.
        """,
    },
    {
        "fuente": "procedimiento_onboarding.md",
        "area": "recursos humanos",
        "fecha": "2026-01-28",
        "texto": """
        Durante el onboarding, el nuevo colaborador debe completar la capacitación de seguridad, aceptar el código de conducta y configurar autenticación multifactor. El líder del equipo debe asignar un mentor y revisar los accesos durante la primera semana. Las solicitudes de permisos adicionales deben justificarse en el sistema de gestión de identidades.
        """,
    },
]

Podemos mirar los documentos como una tabla. Esto no es necesario para RAG, pero resulta útil para recordar que el índice no debería perder información de procedencia.

In [ ]:
import pandas as pd

pd.DataFrame(documentos_empresa)[["fuente", "area", "fecha"]]

## Chunking con metadatos

La pregunta sería: ¿por qué no indexar cada documento completo? En documentos largos, un único embedding puede mezclar muchas ideas. Por eso dividimos cada documento en chunks más pequeños y con solapamiento. El solapamiento ayuda a no cortar una explicación justo en el límite entre dos fragmentos.

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

documentos_llama = [
    Document(
        text=registro["texto"].strip(),
        metadata={
            "fuente": registro["fuente"],
            "area": registro["area"],
            "fecha": registro["fecha"],
        },
    )
    for registro in documentos_empresa
]

splitter = SentenceSplitter(chunk_size=80, chunk_overlap=20)
nodos = splitter.get_nodes_from_documents(documentos_llama)

len(nodos)

In [ ]:
chunks = pd.DataFrame(
    [
        {
            "fuente": nodo.metadata["fuente"],
            "area": nodo.metadata["area"],
            "texto": nodo.get_content(metadata_mode="none"),
        }
        for nodo in nodos
    ]
)

chunks.head()

## Creando el índice vectorial

Ahora configuramos un modelo de embeddings multilingüe y construimos un índice vectorial. En este notebook el índice vive en memoria. En una implementación empresarial, el mismo patrón suele conectarse a un vector database o a un motor de búsqueda híbrida para persistencia, escalabilidad y control operativo.

In [ ]:
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
Settings.llm = None

indice = VectorStoreIndex(nodos)
retriever = indice.as_retriever(similarity_top_k=3)

## Recuperando evidencia

Antes de pedirle al modelo que redacte una respuesta, inspeccionemos qué recupera el índice. Este paso es clave: si el retrieval trae mala evidencia, el generador tendrá muy pocas chances de responder bien.

In [ ]:
pregunta = "¿Qué controles recomienda la política para usar datos sensibles en un proyecto de IA?"
nodos_recuperados = retriever.retrieve(pregunta)

pd.DataFrame(
    [
        {
            "score": round(resultado.score or 0, 3),
            "fuente": resultado.metadata["fuente"],
            "area": resultado.metadata["area"],
            "texto": resultado.node.get_content(metadata_mode="none"),
        }
        for resultado in nodos_recuperados
    ]
)

## Conectando un SLM para responder

Usaremos un modelo pequeño de tipo text-to-text. No esperamos la misma calidad que en un LLM grande, pero resulta suficiente para demostrar la separación de responsabilidades: el índice trae evidencia y el SLM redacta una respuesta condicionada por esa evidencia.

In [ ]:
from transformers import pipeline

modelo_slm = "google/flan-t5-small"
generador = pipeline(
    "text2text-generation",
    model=modelo_slm,
    tokenizer=modelo_slm,
)

def construir_contexto(resultados, max_chars=1800):
    partes = []
    usados = 0
    for idx, resultado in enumerate(resultados, start=1):
        texto = resultado.node.get_content(metadata_mode="none").strip()
        fuente = resultado.metadata["fuente"]
        area = resultado.metadata["area"]
        bloque = f"[Fragmento {idx} | fuente={fuente} | area={area}]\n{texto}"
        if usados + len(bloque) > max_chars:
            break
        partes.append(bloque)
        usados += len(bloque)
    return "\n\n".join(partes)

def responder_con_rag(pregunta, umbral=0.2):
    resultados = retriever.retrieve(pregunta)
    evidencia = [resultado for resultado in resultados if (resultado.score or 0) >= umbral]

    if not evidencia:
        return "No encontré evidencia suficiente en los documentos recuperados.", resultados

    contexto = construir_contexto(evidencia)
    prompt = f"""
Responda en español usando solamente el contexto. Si el contexto no contiene la respuesta, diga que no hay evidencia suficiente.

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:
"""
    respuesta = generador(prompt, max_new_tokens=180, do_sample=False)[0]["generated_text"]
    return respuesta, resultados

Probemos ahora el flujo completo: pregunta, retrieval, prompt y generación.

In [ ]:
respuesta, evidencia = responder_con_rag(pregunta)

print("Pregunta:", pregunta)
print("Respuesta:", respuesta)

También podemos observar las fuentes que sustentan la respuesta. Este tipo de trazabilidad es una diferencia importante entre un chatbot genérico y una solución RAG preparada para contextos más exigentes.

In [ ]:
pd.DataFrame(
    [
        {
            "score": round(resultado.score or 0, 3),
            "fuente": resultado.metadata["fuente"],
            "area": resultado.metadata["area"],
        }
        for resultado in evidencia
    ]
)

## Una pregunta fuera del alcance

Un sistema empresarial no debería inventar respuestas cuando la base documental no contiene evidencia. Un mecanismo simple es revisar el score de recuperación y responder explícitamente cuando no hay soporte suficiente. En producción este control suele complementarse con evaluación, reglas de negocio y monitoreo.

In [ ]:
pregunta_fuera_de_alcance = "¿Cuál es el menú de la cafetería de la oficina?"
respuesta, evidencia = responder_con_rag(pregunta_fuera_de_alcance, umbral=0.35)

print("Pregunta:", pregunta_fuera_de_alcance)
print("Respuesta:", respuesta)

## ¿Qué faltaría para producción?

Este ejemplo sigue siendo pequeño, pero ya muestra las piezas principales de una arquitectura RAG más robusta:

- Ingesta separada de la consulta en línea.
- Chunking explícito con solapamiento.
- Conservación de metadatos de procedencia.
- Índice vectorial para retrieval semántico.
- SLM desacoplado del índice para generar la respuesta.
- Guardas simples para evitar respuestas sin evidencia.

En un escenario real agregaríamos persistencia del índice, control de permisos por documento, versionado, monitoreo de latencia, evaluación de respuestas, reranking, búsqueda híbrida y pruebas con preguntas representativas del dominio.

## Cierre

RAG no es solo llamar a un LLM con un texto adicional. Podemos pensarlo como un pipeline de información: preparar documentos, representarlos como embeddings, recuperar evidencia, construir un prompt controlado y generar una respuesta verificable. La calidad final depende tanto del generador como del índice y de la forma en que partimos los documentos.